# 03 — Embeddings, Chroma, and RAG

**Goal:** understand retrieval before generation.

```text
document → chunks → embeddings → Chroma
question → embedding → nearest chunks → grounded prompt → answer + citations
```

## Chunking

A large document is divided into searchable pieces. The current project uses deterministic fixed-word chunks. It is a simple baseline, not automatically the best strategy.

In [ ]:
def chunk_text(text: str, chunk_size: int) -> list[str]:
    if not text.strip():
        raise ValueError("Text cannot be empty")
    if chunk_size <= 0:
        raise ValueError("Chunk size must be greater than 0")

    words = text.split()
    return [
        " ".join(words[index:index + chunk_size])
        for index in range(0, len(words), chunk_size)
    ]


print(chunk_text("one two three four five", chunk_size=2))

## Embeddings and distance

An embedding model converts text meaning into a vector. Chroma stores document vectors and searches for vectors close to the question vector.

For cosine **distance**, smaller is closer. Distance is not a confidence percentage.

In [ ]:
from math import sqrt


def cosine_distance(left: list[float], right: list[float]) -> float:
    dot_product = sum(a * b for a, b in zip(left, right))
    left_length = sqrt(sum(value * value for value in left))
    right_length = sqrt(sum(value * value for value in right))
    similarity = dot_product / (left_length * right_length)
    return 1 - similarity


question_vector = [1.0, 0.0]
chunks = [
    {"id": "chunk-0", "text": "FastAPI exposes an endpoint.", "vector": [0.0, 1.0]},
    {"id": "chunk-1", "text": "Chroma stores embeddings.", "vector": [1.0, 0.0]},
]

ranked = sorted(
    chunks,
    key=lambda chunk: cosine_distance(question_vector, chunk["vector"]),
)
print([chunk["id"] for chunk in ranked])

## Grounding and citations

Retrieved chunks become the context supplied to the model. The prompt tells the model to use only that evidence. A retrieval citation proves that a chunk was supplied to the model; it does not yet prove that every generated sentence is supported.

In [ ]:
retrieval = {
    "ids": ["project_notes-1"],
    "texts": ["Chroma stores embeddings."],
    "distances": [0.08],
    "metadatas": [{"source": "project_notes.txt", "chunk_index": 1}],
}

citations = [
    {
        "source": metadata["source"],
        "chunk_id": chunk_id,
        "chunk_index": metadata.get("chunk_index"),
        "distance": distance,
    }
    for chunk_id, distance, metadata in zip(
        retrieval["ids"], retrieval["distances"], retrieval["metadatas"]
    )
]

print(citations)

## YOUR TURN

1. Change the question vector so `chunk-0` ranks first.
2. Add a second citation record and inspect the shape.
3. Explain why Chroma can still return a nearest chunk for an unrelated question.
4. Explain the difference between retrieval failure and generation failure.